In [ ]:
!pip install transformers pyarrow tqdm


In [ ]:
import torch
print(torch.cuda.is_available())

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
import torchvision.transforms as transforms
from torch.utils.data import Dataset, DataLoader
import pandas as pd
import io
import base64
from PIL import Image
import numpy as np
import time  # For progress tracking
import os
import matplotlib.pyplot as plt
import torchvision  # Add this import for torchvision utilities
import torch.nn.functional as F

# Additional imports for text processing
from torch.nn.utils.rnn import pad_sequence
from torchvision.utils import make_grid
from tqdm import tqdm

# Import tokenizer and embedding model
from transformers import AutoTokenizer, AutoModelForTokenClassification

# Increase the maximum prompt length to handle detailed descriptions
MAX_PROMPT_LENGTH = 100  # Adjust as needed based on prompt lengths

# Initialize tokenizer and embedder globally
tokenizer = AutoTokenizer.from_pretrained("dslim/bert-base-NER")
embedder = AutoModelForTokenClassification.from_pretrained("dslim/bert-base-NER", output_hidden_states=True)
embedder.eval()

# Move embedder to device when needed
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
embedder.to(device)

# Update the embedding dimension since bert-base has different dimensions
text_embed_dim = 768  # bert-base hidden size is 768 instead of 1024

# Custom Dataset class for Parquet file
class ParquetImageDataset(Dataset):
    def __init__(self, parquet_path, transform=None):
        self.df = pd.read_parquet(parquet_path)
        self.transform = transform
        # Clean the prompts in the dataset
        self.df['Prompt'] = self.df['Prompt'].apply(self.clean_prompt)

    def clean_prompt(self, prompt):
        """Clean the prompt by removing outfit and style information"""
        # Split the prompt into parts
        parts = prompt.split(',')
        # Keep only the relevant parts (remove outfit and style information)
        cleaned_parts = []
        for part in parts:
            # Skip parts containing outfit or style information
            if not any(skip in part.lower() for skip in ['wearing', 'outfit', 'styled as']):
                cleaned_parts.append(part.strip())
        # Join the parts back together
        cleaned_prompt = ', '.join(cleaned_parts)
        return cleaned_prompt

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        # Get base64 image
        image_data = base64.b64decode(self.df.iloc[idx]['Image'])
        image = Image.open(io.BytesIO(image_data)).convert('RGB')

        if self.transform:
            image = self.transform(image)

        # Tokenize and embed the prompt
        prompt = self.df.iloc[idx]['Prompt']
        tokens = tokenizer(
            prompt,
            return_tensors='pt',
            padding='max_length',
            truncation=True,
            max_length=MAX_PROMPT_LENGTH
        ).to(device)

        with torch.no_grad():
            outputs = embedder(**tokens)
            # Get the last hidden state directly
            hidden_states = outputs.hidden_states[-1]
            # Mean pooling to get a single vector
            prompt_embedding = hidden_states.mean(dim=1).squeeze(0)

        return image, prompt_embedding

# Generator Network
class Generator(nn.Module):
    def __init__(self, latent_dim, text_embed_dim):
        super(Generator, self).__init__()
        self.latent_dim = latent_dim
        self.text_embed_dim = text_embed_dim

        # Improved initial processing of noise and text
        self.noise_proj = nn.Linear(latent_dim, 256)
        self.text_proj = nn.Linear(text_embed_dim, 256)
        self.combined_proj = nn.Linear(512, 8*8*512)

        self.main = nn.Sequential(
            # Initial shape: 512 x 8 x 8
            nn.BatchNorm2d(512),
            nn.LeakyReLU(0.2, True),
            # 512 x 8 x 8
            nn.ConvTranspose2d(512, 256, 4, 2, 1, bias=False),
            nn.BatchNorm2d(256),
            nn.LeakyReLU(0.2, True),
            # 256 x 16 x 16
            nn.ConvTranspose2d(256, 128, 4, 2, 1, bias=False),
            nn.BatchNorm2d(128),
            nn.LeakyReLU(0.2, True),
            # 128 x 32 x 32
            nn.ConvTranspose2d(128, 64, 4, 2, 1, bias=False),
            nn.BatchNorm2d(64),
            nn.LeakyReLU(0.2, True),
            # 64 x 64 x 64
            nn.ConvTranspose2d(64, 3, 3, 1, 1, bias=False),
            nn.Tanh()
        )

    def forward(self, noise, text_embedding):
        # Process noise and text separately then combine
        noise_feat = self.noise_proj(noise)
        text_feat = self.text_proj(text_embedding)
        combined = torch.cat((noise_feat, text_feat), dim=1)
        x = F.leaky_relu(self.combined_proj(combined), 0.2)
        x = x.view(-1, 512, 8, 8)
        x = self.main(x)
        return x

# Discriminator Network
class Discriminator(nn.Module):
    def __init__(self, text_embed_dim):
        super(Discriminator, self).__init__()
        self.text_embed_dim = text_embed_dim

        # Improved text processing
        self.text_proj = nn.Sequential(
            nn.Linear(text_embed_dim, 256),
            nn.LeakyReLU(0.2, True),
            nn.Linear(256, 512),
            nn.LeakyReLU(0.2, True)
        )

        self.main = nn.Sequential(
            # Input: 3 x 64 x 64
            nn.Conv2d(3, 64, 4, 2, 1, bias=False),
            nn.LeakyReLU(0.2, inplace=True),
            # 64 x 32 x 32
            nn.Conv2d(64, 128, 4, 2, 1, bias=False),
            nn.BatchNorm2d(128),
            nn.LeakyReLU(0.2, inplace=True),
            # 128 x 16 x 16
            nn.Conv2d(128, 256, 4, 2, 1, bias=False),
            nn.BatchNorm2d(256),
            nn.LeakyReLU(0.2, inplace=True),
            # 256 x 8 x 8
            nn.Conv2d(256, 512, 4, 2, 1, bias=False),
            nn.BatchNorm2d(512),
            nn.LeakyReLU(0.2, inplace=True),
        )

        # Improved final layers
        self.final = nn.Sequential(
            nn.Linear(512*4*4 + 512, 1024),
            nn.LeakyReLU(0.2, True),
            nn.Dropout(0.5),
            nn.Linear(1024, 1)
        )

    def forward(self, image, text_embedding):
        img_features = self.main(image)
        img_features = img_features.view(-1, 512*4*4)
        text_features = self.text_proj(text_embedding)
        combined = torch.cat((img_features, text_features), dim=1)
        x = self.final(combined)
        x = torch.sigmoid(x)
        return x.squeeze()

# Function to initialize weights
def weights_init(m):
    classname = m.__class__.__name__
    if classname.find('Conv') != -1 or classname.find('Linear') != -1:
        nn.init.xavier_normal_(m.weight.data)

def train_gan(
    parquet_path, num_epochs=100, batch_size=16, latent_dim=100, text_embed_dim=768, save_dir='/kaggle/working/',
    start_epoch=0, checkpoint_path=None
):
    # Create save directory if it doesn't exist
    os.makedirs(save_dir, exist_ok=True)
    samples_dir = os.path.join(save_dir, 'training_samples')
    os.makedirs(samples_dir, exist_ok=True)
    checkpoints_dir = os.path.join(save_dir, 'checkpoints')
    os.makedirs(checkpoints_dir, exist_ok=True)

    # Set device
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print(f"Using device: {device}")

    # Adjust the max length in the tokenizer throughout the code
    global MAX_PROMPT_LENGTH

    # Initialize transforms
    transform = transforms.Compose([
        transforms.Resize((64, 64)),
        transforms.ToTensor(),
        transforms.Normalize((0.5,), (0.5,))
    ])

    # Create dataset and dataloader
    dataset = ParquetImageDataset(parquet_path, transform=transform)
    dataloader = DataLoader(dataset, batch_size=batch_size, shuffle=True, num_workers=0)

    # Initialize networks
    generator = Generator(latent_dim, text_embed_dim).to(device)
    discriminator = Discriminator(text_embed_dim).to(device)
    generator.apply(weights_init)
    discriminator.apply(weights_init)

    # Setup optimizers with adjusted learning rates
    g_optimizer = optim.Adam(generator.parameters(), lr=0.0002, betas=(0.5, 0.999))
    d_optimizer = optim.Adam(discriminator.parameters(), lr=0.0002, betas=(0.5, 0.999))

    # Loss function
    criterion = nn.BCELoss()

    # Lists to keep track of progress
    G_losses = []
    D_losses = []
    iters = 0
    num_fixed_samples = 2  # Match the number of fixed prompts
    fixed_noise = torch.randn(num_fixed_samples, latent_dim, device=device)

    # Updated fixed prompts for generating samples
    fixed_prompts = [
        "A adult male, from North India, with a diamond face shape, medium black hair styled as straight, black eyes, with a straight nose, with thin lips, with thick eyebrows, with medium ears.",
        "A adult male, from North India, with a diamond face shape, medium black hair styled as straight, black eyes, with a straight nose, with medium lips, with straight eyebrows, with small ears."
    ]

    # Process fixed prompts
    fixed_text_embeddings = []
    for prompt in fixed_prompts:
        tokens = tokenizer(
            prompt,
            return_tensors='pt',
            padding='max_length',
            truncation=True,
            max_length=MAX_PROMPT_LENGTH
        ).to(device)
        with torch.no_grad():
            outputs = embedder(**tokens)
            # Get the last hidden state directly
            hidden_states = outputs.hidden_states[-1]
            # Mean pooling to get a single vector
            embedding = hidden_states.mean(dim=1).squeeze(0)
            fixed_text_embeddings.append(embedding)
    fixed_text_embeddings = torch.stack(fixed_text_embeddings)

    # Load from checkpoint if provided
    if checkpoint_path and os.path.exists(checkpoint_path):
        print(f"Loading checkpoint from {checkpoint_path}")
        checkpoint = torch.load(checkpoint_path)
        generator.load_state_dict(checkpoint['generator_state'])
        discriminator.load_state_dict(checkpoint['discriminator_state'])
        g_optimizer.load_state_dict(checkpoint['g_optimizer_state'])
        d_optimizer.load_state_dict(checkpoint['d_optimizer_state'])
        start_epoch = checkpoint['epoch']
        G_losses = checkpoint.get('G_losses', [])
        D_losses = checkpoint.get('D_losses', [])
        iters = checkpoint.get('iters', 0)
        print(f"Resuming from epoch {start_epoch}")
    else:
        G_losses = []
        D_losses = []
        iters = 0

    # Training loop
    print(f"Starting training for {num_epochs} epochs from epoch {start_epoch}...")
    for epoch in range(start_epoch, num_epochs):
        print(f"\nEpoch {epoch+1}/{num_epochs}")
        start_time = time.time()
        total_batches = len(dataloader)

        for i, (real_images, prompt_embeddings) in enumerate(dataloader):
            batch_size = real_images.size(0)

            # Move tensors to device
            real_images = real_images.to(device)
            prompt_embeddings = prompt_embeddings.to(device)

            # Ensure proper dimensions for prompt embeddings
            if len(prompt_embeddings.shape) == 1:
                prompt_embeddings = prompt_embeddings.unsqueeze(0)

            # Train Discriminator
            discriminator.zero_grad()

            # Add noise to labels for label smoothing
            label_real = torch.full((batch_size,), 0.9, device=device)  # Changed from 1.0 to 0.9
            label_fake = torch.full((batch_size,), 0.1, device=device)  # Changed from 0.0 to 0.1

            # Add noise to real images for regularization
            real_images = real_images + 0.05 * torch.randn_like(real_images)

            # Generate fake images
            noise = torch.randn(batch_size, latent_dim, device=device)
            fake_images = generator(noise, prompt_embeddings)

            # Train with real images
            output_real = discriminator(real_images, prompt_embeddings)
            d_loss_real = criterion(output_real, label_real)

            # Train with fake images
            output_fake = discriminator(fake_images.detach(), prompt_embeddings)
            d_loss_fake = criterion(output_fake, label_fake)

            d_loss = d_loss_real + d_loss_fake
            d_loss.backward()
            d_optimizer.step()

            # Train Generator
            generator.zero_grad()
            output_fake = discriminator(fake_images, prompt_embeddings)
            g_loss = criterion(output_fake, label_real)
            g_loss.backward()
            g_optimizer.step()

            # Save Losses
            G_losses.append(g_loss.item())
            D_losses.append(d_loss.item())

            # Update progress information
            if (i + 1) % 10 == 0:  # Print every 10 batches
                elapsed = time.time() - start_time
                progress = (i + 1) / total_batches
                print(f"\rBatch {i+1}/{total_batches} "
                      f"({progress:.1%}) - "
                      f"D_loss: {d_loss.item():.4f} "
                      f"G_loss: {g_loss.item():.4f} "
                      f"Time: {elapsed:.1f}s",
                      end="")

            # Generate and save sample images
            if iters % 100 == 0:
                with torch.no_grad():
                    fake = generator(fixed_noise, fixed_text_embeddings)
                    fake = (fake * 0.5) + 0.5  # Denormalize
                    grid = make_grid(fake.cpu(), nrow=2)
                    plt.figure(figsize=(8,8))
                    plt.axis("off")
                    plt.title(f"Epoch {epoch+1}, Iteration {iters}")
                    plt.imshow(np.transpose(grid.numpy(), (1,2,0)))
                    plt.savefig(f"{samples_dir}/epoch_{epoch+1}_iter_{iters}.png")
                    plt.close()
                print(f"\nSaved samples at iteration {iters}")

            iters += 1

        # Save checkpoint every 100 epochs
        if (epoch + 1) % 5 == 0:
            checkpoint = {
                'epoch': epoch + 1,
                'generator_state': generator.state_dict(),
                'discriminator_state': discriminator.state_dict(),
                'g_optimizer_state': g_optimizer.state_dict(),
                'd_optimizer_state': d_optimizer.state_dict(),
                'G_losses': G_losses,
                'D_losses': D_losses,
                'iters': iters,
                'latent_dim': latent_dim,
                'text_embed_dim': text_embed_dim
            }
            checkpoint_path = os.path.join(checkpoints_dir, f'checkpoint_epoch_{epoch+1}.pth')
            torch.save(checkpoint, checkpoint_path)
            print(f"\nSaved checkpoint at epoch {epoch+1}")

        # Print epoch summary
        print(f"\nEpoch {epoch+1} completed in {time.time() - start_time:.1f}s")

    # Save final models
    model_info = {
        'generator_state': generator.state_dict(),
        'discriminator_state': discriminator.state_dict(),
        'latent_dim': latent_dim,
        'text_embed_dim': text_embed_dim,
        'epoch': num_epochs
    }
    torch.save(model_info, os.path.join(save_dir, 'gan_model_final.pth'))

    # Plot and save the loss curves
    plt.figure(figsize=(10,5))
    plt.title("Generator and Discriminator Loss During Training")
    plt.plot(G_losses, label="G Loss")
    plt.plot(D_losses, label="D Loss")
    plt.xlabel("Iterations")
    plt.ylabel("Loss")
    plt.legend()
    plt.savefig(os.path.join(save_dir, 'loss_curves.png'))
    plt.close()

def load_gan_model(model_path):
    """Load a saved GAN model"""
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

    # Load the saved model info
    model_info = torch.load(model_path, map_location=device)

    # Initialize the generator with the same latent and text embedding dimensions
    generator = Generator(model_info['latent_dim'], model_info['text_embed_dim']).to(device)
    generator.load_state_dict(model_info['generator_state'])
    generator.eval()  # Set to evaluation mode

    # No need to reload tokenizer and embedder here since they are already loaded globally

    return generator, model_info['latent_dim'], model_info['text_embed_dim']

def generate_images_from_prompts(generator, latent_dim, prompts, device=None):
    """Generate images conditioned on prompts using the trained generator"""
    if device is None:
        device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

    generator.eval()
    with torch.no_grad():
        # Generate random noise
        noise = torch.randn(len(prompts), latent_dim, device=device)

        # Embed prompts
        text_embeddings = []
        for prompt in prompts:
            tokens = tokenizer(
                prompt,
                return_tensors='pt',
                padding='max_length',
                truncation=True,
                max_length=MAX_PROMPT_LENGTH
            ).to(device)

            # Get embeddings using hidden states
            outputs = embedder(**tokens)
            # Get the last hidden state
            hidden_states = outputs.hidden_states[-1]
            # Mean pooling to get a single vector
            embedding = hidden_states.mean(dim=1).squeeze(0)
            text_embeddings.append(embedding)

        text_embeddings = torch.stack(text_embeddings)

        # Generate images
        generated_images = generator(noise, text_embeddings)

        # Convert images from tensor to PIL Image
        generated_images = (generated_images * 0.5 + 0.5).clamp(0, 1)
        images = []
        for img_tensor in generated_images.cpu():
            img_array = img_tensor.permute(1, 2, 0).numpy()
            img = Image.fromarray((img_array * 255).astype('uint8'))
            images.append(img)

        return images

def generate_and_save_images_from_prompts(model_path, prompts, output_dir='generated_images'):
    """Load model, generate images from prompts, and save them"""
    # Create output directory if it doesn't exist
    os.makedirs(output_dir, exist_ok=True)

    # Load the model
    generator, latent_dim, _ = load_gan_model(model_path)
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

    # Generate images
    images = generate_images_from_prompts(generator, latent_dim, prompts, device=device)

    # Save images
    for i, img in enumerate(images):
        img.save(os.path.join(output_dir, f'generated_image_{i+1}.png'))

    return images

if __name__ == "__main__":
    import os

    # Training
    parquet_path = "/content/dataset/GAN-Training Dataset/combined_file.parquet"
    save_dir = '/content/output/'

    # First, train the model
    print("Training GAN...")
    # Check for latest checkpoint
    checkpoints_dir = os.path.join(save_dir, 'checkpoints')
    if os.path.exists(checkpoints_dir):
        checkpoints = sorted([f for f in os.listdir(checkpoints_dir) if f.startswith('checkpoint_epoch_')])
        if checkpoints:
            latest_checkpoint = os.path.join(checkpoints_dir, checkpoints[-1])
            print(f"Found checkpoint: {latest_checkpoint}")
            train_gan(
                parquet_path,
                num_epochs=100,
                batch_size=16,
                latent_dim=100,
                text_embed_dim=768,
                save_dir=save_dir,
                checkpoint_path=latest_checkpoint
            )
        else:
            print("No checkpoint found, starting fresh training")
            train_gan(
                parquet_path,
                num_epochs=100,
                batch_size=16,
                latent_dim=100,
                text_embed_dim=768,
                save_dir=save_dir
            )
    else:
        print("No checkpoints directory found, starting fresh training")
        train_gan(
            parquet_path,
            num_epochs=100,
            batch_size=16,
            latent_dim=100,
            text_embed_dim=768,
            save_dir=save_dir
        )

    # Then generate some images using the trained model
    print("\nGenerating images from trained model...")
    model_path = os.path.join(save_dir, 'gan_model_final.pth')
    output_dir = 'generated_images'

    # Define prompts for image generation
    prompts = [
        "A adult male, from North India, with a diamond face shape, medium black hair styled as straight, black eyes, with a straight nose, with thin lips, with thick eyebrows, with medium ears.",
        "A adult male, from North India, with a diamond face shape, medium black hair styled as straight, black eyes, with a straight nose, with medium lips, with straight eyebrows, with small ears.",
        "A adult male, from North India, with an oval face shape, short black hair styled as straight, black eyes, with a straight nose, with medium lips, with thick eyebrows, with small ears."
    ]

    # Generate and save images from prompts
    generated_images = generate_and_save_images_from_prompts(model_path, prompts, output_dir=output_dir)
    print(f"Generated images have been saved to {output_dir}/")

In [ ]:
import os

print(os.listdir('/content/output/training_samples'))

In [ ]:
from PIL import Image
import matplotlib.pyplot as plt

img1 = Image.open('/content/output/training_samples/epoch_1_iter_0.png')

plt.figure(figsize=(8,8))
plt.imshow(img1)
plt.axis('off')
plt.show()

In [ ]:
from PIL import Image
import matplotlib.pyplot as plt

img2 = Image.open('/content/output/training_samples/epoch_1_iter_100.png')

plt.figure(figsize=(8,8))
plt.imshow(img2)
plt.axis('off')
plt.show()

In [ ]:
import zipfile
import os

zip_path = "/content/GAN-Training Dataset-20260508T062440Z-3-001.zip"
extract_path = "/content/dataset"

os.makedirs(extract_path, exist_ok=True)

with zipfile.ZipFile(zip_path, 'r') as zip_ref:
    zip_ref.extractall(extract_path)

print("Dataset extracted successfully!")